# SignBridge ASL Gesture Training (Google Colab)
Run this notebook to quickly train your ASL gesture recognition model on your LOCAL DATA.
Once done, download the `.task` file and place it in your frontend's `public` folder.

In [ ]:
!sudo apt-get install python3.10 python3.10-distutils
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10
!python3.10 -m pip install mediapipe-model-maker==0.2.1.4

In [ ]:
import os
import shutil
import subprocess
import sys

# Using a raw string to prevent any newline syntax errors
code = r"""
import os
os.environ['MPLBACKEND'] = 'Agg'

import shutil
import zipfile
import sys
from mediapipe_model_maker import gesture_recognizer

zip_path = "/content/reduced_dataset.zip"
data_dir = "/content/asl_dataset"

if not os.path.exists(zip_path):
    print("\n=========================================")
    print("CRITICAL ERROR: MISSING DATASET!")
    print("Please upload 'reduced_dataset.zip' from your Desktop")
    print("using the folder icon on the left menu of Colab.")
    print("=========================================\n")
    sys.exit(1)

print("\nExtracting your custom uploaded dataset... (Manual OS Extraction)")
# Robust Windows-to-Linux Zip Extraction
with zipfile.ZipFile(zip_path, 'r') as z:
    for info in z.infolist():
        # Normalize Windows backslashes to Linux forward slashes
        extracted_path = info.filename.replace('\\', '/')
        if extracted_path.startswith("reduced_dataset/"):
            extracted_path = extracted_path[len("reduced_dataset/"):]
        
        # Skip empty names or root paths
        if not extracted_path:
            continue
            
        target_path = os.path.join(data_dir, extracted_path)
        
        # Create directories
        if info.filename.endswith('/') or info.filename.endswith('\\'):
            os.makedirs(target_path, exist_ok=True)
            continue
            
        os.makedirs(os.path.dirname(target_path), exist_ok=True)
        
        # Write the actual file
        with z.open(info) as source, open(target_path, "wb") as target:
            shutil.copyfileobj(source, target)

extracted_path = data_dir
print(f"\nLoading data from {extracted_path}...")
data = gesture_recognizer.Dataset.from_folder(
    dirname=extracted_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

print("\nTraining Neural Network... (Will be fast with Colab GPU!)")
hparams = gesture_recognizer.HParams(export_dir="custom_model_output", epochs=15, batch_size=32)
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)

model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

loss, acc = model.evaluate(test_data, batch_size=1)
print(f"\nTest accuracy: {acc * 100:.2f}%")

model.export_model()
print("\nExported successfully!")
"""
with open('run_training.py', 'w') as f:
    f.write(code)

process = subprocess.run(["python3.10", "run_training.py"], capture_output=True, text=True)
print(process.stdout)
if process.returncode != 0:
   print("ERROR:")
   print(process.stderr)


In [ ]:
import os
from google.colab import files
if os.path.exists('custom_model_output/gesture_recognizer.task'):
    files.download('custom_model_output/gesture_recognizer.task')
else:
    print("Error: task file not found. Check training step above.")